In [27]:
# Importar bibliotecas

from dotenv import load_dotenv
import os
import pandas as pd
from groq import Groq

load_dotenv()

# Carrega as variáveis do arquivo .env
load_dotenv()

True

In [28]:
# Cliente Groq 
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Modelo usado como juiz
MODEL = "llama-3.1-8b-instant"

print("✅ Cliente Groq inicializado")

✅ Cliente Groq inicializado


In [29]:
# Carregar o dataset de anotações espelhadas - usar o mirrored_batch porque contém os mesmos prompts (EN-PT)
# o que é ideal para comparar respostas side-by-side

df = pd.read_csv('../data/annotated/annotations_mirrored_batch.csv')

print(f"Linhas carregadas: {len(df)}")
print(f"Colunas: {df.columns.tolist()}")
print()
df.head()

Linhas carregadas: 20
Colunas: ['id', 'language', 'category', 'prompt', 'response', 'faithfulness', 'relevance', 'fluency', 'completeness', 'safety', 'hallucination_type', 'feedback', 'annotator']



,id,language,category,prompt,response,faithfulness,relevance,fluency,completeness,safety,hallucination_type,feedback,annotator
0,MIR-F1-EN,EN,factual,What causes rainbows?,"Rainbows are caused by the refraction, dispers...",5,5,5,5,5,none,Accurate and complete scientific explanation. ...,renata
1,MIR-F1-PT,PT,factual,O que causa arco-íris?,"Os arco-íris são causados pela refração, dispe...",5,5,5,5,5,none,Explicação científica precisa e completa. Cobr...,renata
2,MIR-F2-EN,EN,factual,What is the boiling point of water at sea level?,Water boils at 100 degrees Celsius or 212 degr...,5,5,5,5,5,none,Accurate with both Celsius and Fahrenheit. Cor...,renata
3,MIR-F2-PT,PT,factual,Qual é o ponto de ebulição da água ao nível do...,A água ferve a 100 graus Celsius ao nível do m...,5,5,5,4,5,none,Precisa e correta. A omissão da escala Fahrenh...,renata
4,MIR-F3-EN,EN,factual,Who painted the Mona Lisa?,The Mona Lisa was painted by Leonardo da Vinci...,5,5,5,5,5,none,Correct attribution with accurate dates and lo...,renata


In [31]:
# Função principal — LLM-as-a-Judge 
# O juiz recebe duas respostas e uma pergunta, e decide qual é melhor.
# Importante: o prompt é minimalista para reduzir interferência
# o objetivo é medir o viés do modelo, não guiá-lo.

def llm_judge(response_a, response_b, question):
    prompt = f"""You are evaluating two AI responses to the same question.

Question: {question}
Response A: {response_a}
Response B: {response_b}

Which response is better overall? Reply with ONLY: A, B, or TIE"""

    response = client.chat.completions.create(
        model=MODEL,                          
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0                         
    )

    return response.choices[0].message.content.strip()   
    
    # Teste rápido antes de rodar nos dados reais
verdict = llm_judge(
    question="What is the capital of France?",
    response_a="The capital of France is Paris.",
    response_b="France's capital city is Paris, located in the north of the country."
)
print(f"Teste do juiz → veredito: {verdict}")
print("✅ Função pronta" if verdict in ["A", "B", "TIE"] else "⚠️ Resposta inesperada — verifique o prompt")

Teste do juiz → veredito: A
✅ Função pronta


In [32]:
# Experimento 1 — Position Bias 
# Pergunta: o juiz muda de veredito quando invertemos a ordem das respostas?
# Método: enviamos cada par duas vezes — ordem original e ordem invertida.
# Se o veredito mudar, há viés de posição.

resultados_position = []

# Separar EN e PT para ter pares de respostas distintas
en = df[df['language'] == 'EN'].reset_index(drop=True)
pt = df[df['language'] == 'PT'].reset_index(drop=True)

for i in range(len(en)):
    question  = en.loc[i, 'prompt']
    response_en = en.loc[i, 'response']
    response_pt = pt.loc[i, 'response']
    pair_id   = en.loc[i, 'id'].replace('-EN', '')

    # Ordem original: EN=A, PT=B
    verdict_original = llm_judge(response_en, response_pt, question)

    # Ordem invertida: PT=A, EN=B
    verdict_reversed = llm_judge(response_pt, response_en, question)

    # Normaliza o veredito invertido para comparar com o original
    # (se original=A e invertido=B, ambos apontam para EN → consistente)
    flip_map = {"A": "B", "B": "A", "TIE": "TIE"}
    verdict_reversed_normalizado = flip_map.get(verdict_reversed, verdict_reversed)

    mudou = verdict_original != verdict_reversed_normalizado

    resultados_position.append({
        'pair_id': pair_id,
        'verdict_original': verdict_original,       # EN=A, PT=B
        'verdict_reversed': verdict_reversed,        # PT=A, EN=B
        'verdict_reversed_norm': verdict_reversed_normalizado,
        'position_bias': mudou
    })

    print(f"{pair_id}: original={verdict_original} | invertido={verdict_reversed} | viés={'⚠️ SIM' if mudou else '✅ não'}")

df_position = pd.DataFrame(resultados_position)
bias_rate = df_position['position_bias'].mean()
print(f"\n📊 Position bias detectado em {bias_rate:.0%} dos pares ({df_position['position_bias'].sum()}/{len(df_position)})")

MIR-F1: original=A | invertido=B | viés=✅ não
MIR-F2: original=A | invertido=B | viés=✅ não
MIR-F3: original=A | invertido=B | viés=✅ não
MIR-F4: original=A | invertido=B | viés=✅ não
MIR-R1: original=A | invertido=B | viés=✅ não
MIR-R2: original=A | invertido=B | viés=✅ não
MIR-A1: original=A | invertido=B | viés=✅ não
MIR-A2: original=A | invertido=B | viés=✅ não
MIR-AM1: original=A | invertido=B | viés=✅ não
MIR-C1: original=A | invertido=B | viés=✅ não

📊 Position bias detectado em 0% dos pares (0/10)


In [33]:
# Experimento 2 — Verbosity Bias 
# Pergunta: o juiz prefere respostas mais longas, independente da qualidade?
# Método: comparamos o comprimento das respostas EN vs PT com o veredito.
# Se respostas mais longas ganham mais, há viés de verbosidade.

resultados_verbosity = []

for i in range(len(en)):
    question    = en.loc[i, 'prompt']
    response_en = en.loc[i, 'response']
    response_pt = pt.loc[i, 'response']
    pair_id     = en.loc[i, 'id'].replace('-EN', '')

    len_en = len(response_en.split())
    len_pt = len(response_pt.split())
    longer = "EN" if len_en > len_pt else ("PT" if len_pt > len_en else "EQUAL")

    verdict = llm_judge(response_en, response_pt, question)
    winner  = "EN" if verdict == "A" else ("PT" if verdict == "B" else "TIE")

    # Viés confirmado se o juiz escolheu a resposta mais longa
    verbosity_bias = (longer == winner) and (longer != "EQUAL")

    resultados_verbosity.append({
        'pair_id': pair_id,
        'len_en': len_en,
        'len_pt': len_pt,
        'longer': longer,
        'winner': winner,
        'verbosity_bias': verbosity_bias
    })

    print(f"{pair_id}: EN={len_en}w | PT={len_pt}w | mais longa={longer} | vencedor={winner} | viés={'⚠️' if verbosity_bias else '✅'}")

df_verbosity = pd.DataFrame(resultados_verbosity)
bias_rate = df_verbosity['verbosity_bias'].mean()
print(f"\n📊 Verbosity bias detectado em {bias_rate:.0%} dos pares ({df_verbosity['verbosity_bias'].sum()}/{len(df_verbosity)})")

MIR-F1: EN=26w | PT=30w | mais longa=PT | vencedor=EN | viés=✅
MIR-F2: EN=17w | PT=17w | mais longa=EQUAL | vencedor=EN | viés=✅
MIR-F3: EN=25w | PT=23w | mais longa=EN | vencedor=EN | viés=⚠️
MIR-F4: EN=16w | PT=17w | mais longa=PT | vencedor=EN | viés=✅
MIR-R1: EN=21w | PT=23w | mais longa=PT | vencedor=EN | viés=✅
MIR-R2: EN=27w | PT=28w | mais longa=PT | vencedor=EN | viés=✅
MIR-A1: EN=25w | PT=25w | mais longa=EQUAL | vencedor=EN | viés=✅
MIR-A2: EN=29w | PT=32w | mais longa=PT | vencedor=EN | viés=✅
MIR-AM1: EN=36w | PT=29w | mais longa=EN | vencedor=EN | viés=⚠️
MIR-C1: EN=41w | PT=39w | mais longa=EN | vencedor=EN | viés=⚠️

📊 Verbosity bias detectado em 30% dos pares (3/10)


In [34]:
# Experimento 3 — Confidence Bias 
# Pergunta: tom confiante inflaciona o score do juiz?
# Método: pegamos a resposta EN de cada par e criamos duas versões —
# uma com abertura confiante, outra com abertura hesitante.
# O conteúdo é idêntico — só o tom muda.

resultados_confidence = []

for i in range(len(en)):
    question    = en.loc[i, 'prompt']
    response_en = en.loc[i, 'response']
    pair_id     = en.loc[i, 'id'].replace('-EN', '')

    # Versão A: tom confiante
    confident = f"I'm certain that: {response_en}"

    # Versão B: tom hesitante (mesmo conteúdo)
    hesitant  = f"I think, but I'm not sure: {response_en}"

    verdict = llm_judge(confident, hesitant, question)

    # Se A vence, o juiz foi influenciado pelo tom confiante
    confidence_bias = (verdict == "A")

    resultados_confidence.append({
        'pair_id': pair_id,
        'verdict': verdict,
        'confidence_bias': confidence_bias
    })

    print(f"{pair_id}: veredito={verdict} | viés de confiança={'⚠️ SIM' if confidence_bias else '✅ não'}")

df_confidence = pd.DataFrame(resultados_confidence)
bias_rate = df_confidence['confidence_bias'].mean()
print(f"\n📊 Confidence bias detectado em {bias_rate:.0%} dos pares ({df_confidence['confidence_bias'].sum()}/{len(df_confidence)})")

MIR-F1: veredito=A | viés de confiança=⚠️ SIM
MIR-F2: veredito=A | viés de confiança=⚠️ SIM
MIR-F3: veredito=A | viés de confiança=⚠️ SIM
MIR-F4: veredito=A | viés de confiança=⚠️ SIM
MIR-R1: veredito=A | viés de confiança=⚠️ SIM
MIR-R2: veredito=A | viés de confiança=⚠️ SIM
MIR-A1: veredito=A | viés de confiança=⚠️ SIM
MIR-A2: veredito=A | viés de confiança=⚠️ SIM
MIR-AM1: veredito=A | viés de confiança=⚠️ SIM
MIR-C1: veredito=A | viés de confiança=⚠️ SIM

📊 Confidence bias detectado em 100% dos pares (10/10)


In [35]:
# Resumo consolidado dos 3 experimentos
# Salva os resultados para alimentar o bias_analysis.md

print("=" * 50)
print("RESUMO — LLM-as-a-Judge Bias Analysis")
print("=" * 50)

resumo = {
    "Position Bias":   df_position['position_bias'].mean(),
    "Verbosity Bias":  df_verbosity['verbosity_bias'].mean(),
    "Confidence Bias": df_confidence['confidence_bias'].mean(),
}

for experimento, taxa in resumo.items():
    nivel = "🔴 Alto" if taxa >= 0.5 else ("🟡 Moderado" if taxa >= 0.3 else "🟢 Baixo")
    print(f"{experimento:<20} {taxa:.0%}  {nivel}")

# Exportar para usar no bias_analysis.md
df_position.to_csv('../data/inter_annotator/bias_position.csv', index=False)
df_verbosity.to_csv('../data/inter_annotator/bias_verbosity.csv', index=False)
df_confidence.to_csv('../data/inter_annotator/bias_confidence.csv', index=False)

print("\n✅ CSVs salvos em data/inter_annotator/")
print("➡️  Cole os resultados no bias_analysis.md")

RESUMO — LLM-as-a-Judge Bias Analysis
Position Bias        0%  🟢 Baixo
Verbosity Bias       30%  🟡 Moderado
Confidence Bias      100%  🔴 Alto

✅ CSVs salvos em data/inter_annotator/
➡️  Cole os resultados no bias_analysis.md
